In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
import tensorflow_datasets as tfds

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import regularizers

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

print("TensorFlow version:", tf.__version__)

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 37

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

AUTOTUNE = tf.data.AUTOTUNE

print("Number of classes:", NUM_CLASSES)
print("Image size:", IMG_SIZE)

In [ ]:
(train_raw, test_raw), ds_info = tfds.load(
    "oxford_iiit_pet",
    split=["train", "test"],
    as_supervised=True,
    with_info=True
)

print(ds_info)

In [ ]:
class_names = ds_info.features["label"].names

print("Number of classes:", len(class_names))
print(class_names[:10])

In [ ]:
total_train = ds_info.splits["train"].num_examples

train_size = int(total_train * 0.8)

train_ds = train_raw.take(train_size)
val_ds = train_raw.skip(train_size)

print("Training images:", train_size)
print("Validation images:", total_train - train_size)
print("Test images:", ds_info.splits["test"].num_examples)

In [ ]:
def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

In [ ]:
train_data = (
    train_ds
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_data = (
    val_ds
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_data = (
    test_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

In [ ]:
plt.figure(figsize=(10, 8))

for images, labels in train_data.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i])
        plt.title(class_names[labels[i]])
        plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def build_model(
    dropout_rate=0.0,
    learning_rate=0.001,
    optimizer_name="adam",
    train_base=False,
    l2_value=0.0,
    use_bn=True
):
    
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet"
    )

    base_model.trainable = train_base

    inputs = keras.Input(shape=(224, 224, 3))

    x = data_augmentation(inputs)
    
    # MobileNetV2 expects its inputs in the range used by preprocess_input
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x * 255.0)

    x = base_model(x, training=False)

    if use_bn:
        x = layers.BatchNormalization()(x)

    x = layers.GlobalAveragePooling2D()(x)

    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)

    if l2_value > 0:
        outputs = layers.Dense(
            NUM_CLASSES,
            activation="softmax",
            kernel_regularizer=regularizers.l2(l2_value)
        )(x)
    else:
        outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = keras.Model(inputs, outputs)

    if optimizer_name == "sgd":
        optimizer = keras.optimizers.SGD(learning_rate=learning_rate)
    elif optimizer_name == "momentum":
        optimizer = keras.optimizers.SGD(
            learning_rate=learning_rate,
            momentum=0.9
        )
    elif optimizer_name == "rmsprop":
        optimizer = keras.optimizers.RMSprop(learning_rate=learning_rate)
    else:
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
baseline_model = build_model(
    dropout_rate=0.0,
    learning_rate=0.001,
    optimizer_name="adam",
    train_base=False
)

baseline_model.summary()

In [ ]:
start_time = time.time()

baseline_history = baseline_model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

baseline_time = time.time() - start_time

print("Training time:", round(baseline_time, 2), "seconds")

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(baseline_history.history["accuracy"], label="Training Accuracy")
plt.plot(baseline_history.history["val_accuracy"], label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Baseline Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(baseline_history.history["loss"], label="Training Loss")
plt.plot(baseline_history.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Baseline Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
def plot_history(history, title):
    plt.figure(figsize=(8, 5))
    plt.plot(history.history["accuracy"], label="Training Accuracy")
    plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history.history["loss"], label="Training Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title + " - Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
def build_initialization_model(initializer):
    
    model = keras.Sequential([
        layers.Input(shape=(224, 224, 3)),

        layers.Conv2D(
            32,
            (3, 3),
            activation="relu",
            kernel_initializer=initializer
        ),
        layers.MaxPooling2D(),

        layers.Conv2D(
            64,
            (3, 3),
            activation="relu",
            kernel_initializer=initializer
        ),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D(),

        layers.Dense(
            128,
            activation="relu",
            kernel_initializer=initializer
        ),

        layers.Dense(
            NUM_CLASSES,
            activation="softmax",
            kernel_initializer=initializer
        )
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
def build_initialization_model(initializer):
    
    model = keras.Sequential([
        layers.Input(shape=(224, 224, 3)),

        layers.Conv2D(
            32,
            (3, 3),
            activation="relu",
            kernel_initializer=initializer
        ),
        layers.MaxPooling2D(),

        layers.Conv2D(
            64,
            (3, 3),
            activation="relu",
            kernel_initializer=initializer
        ),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D(),

        layers.Dense(
            128,
            activation="relu",
            kernel_initializer=initializer
        ),

        layers.Dense(
            NUM_CLASSES,
            activation="softmax",
            kernel_initializer=initializer
        )
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
initializers = {
    "Zero": keras.initializers.Zeros(),
    "Random": keras.initializers.RandomNormal(stddev=0.05),
    "Xavier": keras.initializers.GlorotNormal(),
    "He": keras.initializers.HeNormal()
}

initialization_histories = {}

for name, initializer in initializers.items():
    print("\nTraining with", name, "initialization")

    model = build_initialization_model(initializer)

    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=5,
        verbose=1
    )

    initialization_histories[name] = history

In [ ]:
plt.figure(figsize=(9, 6))

for name, history in initialization_histories.items():
    plt.plot(
        history.history["loss"],
        label=name
    )

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss for Different Initializations")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

for name, history in initialization_histories.items():
    plt.plot(
        history.history["val_accuracy"],
        label=name
    )

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy for Different Initializations")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
initialization_results = []

for name, history in initialization_histories.items():
    best_acc = max(history.history["val_accuracy"])
    final_loss = history.history["loss"][-1]

    initialization_results.append({
        "Initialization": name,
        "Final Training Loss": final_loss,
        "Best Validation Accuracy": best_acc
    })

initialization_df = pd.DataFrame(initialization_results)

initialization_df

In [ ]:
regularization_settings = {
    "No Regularization": {
        "dropout": 0.0,
        "l2": 0.0,
        "bn": False
    },

    "L2": {
        "dropout": 0.0,
        "l2": 0.0001,
        "bn": False
    },

    "Dropout": {
        "dropout": 0.5,
        "l2": 0.0,
        "bn": False
    },

    "Batch Normalization": {
        "dropout": 0.0,
        "l2": 0.0,
        "bn": True
    }
}

In [ ]:
regularization_histories = {}

for name, settings in regularization_settings.items():

    print("\nRunning:", name)

    model = build_model(
        dropout_rate=settings["dropout"],
        l2_value=settings["l2"],
        use_bn=settings["bn"],
        learning_rate=0.001,
        optimizer_name="adam"
    )

    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=5,
        verbose=1
    )

    regularization_histories[name] = history

In [ ]:
plt.figure(figsize=(9, 6))

for name, history in regularization_histories.items():
    plt.plot(
        history.history["val_accuracy"],
        label=name
    )

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Effect of Regularization")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

no_bn = regularization_histories["No Regularization"]
with_bn = regularization_histories["Batch Normalization"]

plt.plot(
    no_bn.history["val_accuracy"],
    label="Without BN"
)

plt.plot(
    with_bn.history["val_accuracy"],
    label="With BN"
)

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("With vs Without Batch Normalization")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
x = np.array([2, 4, 6, 8], dtype=float)

mean = np.mean(x)
variance = np.mean((x - mean) ** 2)

normalized = (x - mean) / np.sqrt(variance)

print("Mean:", mean)
print("Variance:", variance)
print("Normalized values:", normalized)

In [ ]:
optimizer_histories = {}
optimizer_times = {}

optimizers = ["sgd", "momentum", "rmsprop", "adam"]

for optimizer_name in optimizers:

    print("\nTraining with:", optimizer_name)

    model = build_model(
        dropout_rate=0.25,
        learning_rate=0.001,
        optimizer_name=optimizer_name,
        train_base=False
    )

    start = time.time()

    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=5,
        verbose=1
    )

    elapsed = time.time() - start

    optimizer_histories[optimizer_name] = history
    optimizer_times[optimizer_name] = elapsed

In [ ]:
plt.figure(figsize=(9, 6))

for name, history in optimizer_histories.items():
    plt.plot(
        history.history["loss"],
        label=name.upper()
    )

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss for Different Optimizers")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

for name, history in optimizer_histories.items():
    plt.plot(
        history.history["val_accuracy"],
        label=name.upper()
    )

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy for Different Optimizers")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
optimizer_results = []

for name, history in optimizer_histories.items():

    best_val_acc = max(history.history["val_accuracy"])
    final_loss = history.history["loss"][-1]

    optimizer_results.append({
        "Optimizer": name.upper(),
        "Final Loss": final_loss,
        "Best Validation Accuracy": best_val_acc,
        "Training Time (sec)": optimizer_times[name]
    })

optimizer_df = pd.DataFrame(optimizer_results)

optimizer_df

In [ ]:
lr_histories = {}

for lr in learning_rates:

    print("\nLearning rate:", lr)

    model = build_model(
        learning_rate=lr,
        dropout_rate=0.25,
        optimizer_name="adam"
    )

    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=5,
        verbose=1
    )

    lr_histories[lr] = history

In [ ]:
lr_results = []

for lr, history in lr_histories.items():
    best_acc = max(history.history["val_accuracy"])

    lr_results.append({
        "Learning Rate": lr,
        "Best Validation Accuracy": best_acc
    })

lr_df = pd.DataFrame(lr_results)

plt.figure(figsize=(7, 5))
plt.plot(
    lr_df["Learning Rate"],
    lr_df["Best Validation Accuracy"],
    marker="o"
)

plt.xlabel("Learning Rate")
plt.ylabel("Validation Accuracy")
plt.title("Learning Rate vs Validation Accuracy")
plt.grid(True)
plt.show()

lr_df

In [ ]:
dropout_histories = {}

for dropout in dropout_rates:

    print("\nDropout:", dropout)

    model = build_model(
        learning_rate=0.001,
        dropout_rate=dropout,
        optimizer_name="adam"
    )

    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=5,
        verbose=1
    )

    dropout_histories[dropout] = history

In [ ]:
dropout_results = []

for dropout, history in dropout_histories.items():

    best_acc = max(history.history["val_accuracy"])

    dropout_results.append({
        "Dropout Rate": dropout,
        "Best Validation Accuracy": best_acc
    })

dropout_df = pd.DataFrame(dropout_results)

plt.figure(figsize=(7, 5))

plt.plot(
    dropout_df["Dropout Rate"],
    dropout_df["Best Validation Accuracy"],
    marker="o"
)

plt.xlabel("Dropout Rate")
plt.ylabel("Validation Accuracy")
plt.title("Dropout Rate vs Validation Accuracy")
plt.grid(True)
plt.show()

dropout_df

In [ ]:
feature_model = build_model(
    dropout_rate=0.25,
    learning_rate=0.001,
    optimizer_name="adam",
    train_base=False
)

feature_model.summary()

In [ ]:
start = time.time()

feature_history = feature_model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

feature_time = time.time() - start

print("Feature extraction time:", round(feature_time, 2), "seconds")

In [ ]:
fine_tune_model = feature_model

base_model = None

for layer in fine_tune_model.layers:
    if isinstance(layer, tf.keras.Model):
        base_model = layer
        break

print("Base model found:", base_model is not None)

In [ ]:
base_model.trainable = True

# Keep most of the pretrained network frozen
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Keep Batch Normalization layers frozen
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

print("Trainable layers:",
      sum(layer.trainable for layer in base_model.layers))

In [ ]:
start = time.time()

fine_tune_history = fine_tune_model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

fine_tune_time = time.time() - start

print("Fine-tuning time:", round(fine_tune_time, 2), "seconds")